# Evaluation tools - checks and usage

Module 4 (Evaluation & demo). The functions live in `src/evaluation.py` so that every model is scored the same way:

- macro-F1 (primary metric), micro-F1, weighted-F1, exact match, per-class precision / recall / F1
- bootstrap 95% confidence intervals, and a **paired** bootstrap to compare two models on the same comments
- per-class thresholds tuned on the **validation** set (test is used once, at the end)
- scores at the three levels (28 labels, Ekman, sentiment)
- calibration: Brier score, ECE, reliability diagrams, isotonic / Platt recalibration
- robustness suite of hand-written messages (`src/robustness_suite.csv`) with Wilson 95% CIs

This notebook (1) checks that the functions compute the right thing, (2) shows how to plug in a model. Run `01_eda_simple.ipynb` first.

In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.append(os.path.join("..", "src"))
from evaluation import *

DATA_DIR = os.path.join("..", "data", "processed")

## 1. Load the prepared splits as multi-hot matrices

In [ ]:
splits = {name: pd.read_parquet(os.path.join(DATA_DIR, f"{name}.parquet")) for name in ["train", "validation", "test"]}

# Y[split][level] -> 0/1 array, columns in the order of LEVELS[level]
Y = {name: {level: to_multi_hot(df[level], classes) for level, classes in LEVELS.items()}
     for name, df in splits.items()}

for name in splits:
    print(name, {level: y.shape for level, y in Y[name].items()})

## 2. Checks

Each check is an `assert`: if a function is wrong, the cell stops.

In [ ]:
y_val = Y["validation"]["labels_28"]

# 2.1 Regrouping the true 28 labels gives exactly the Ekman / sentiment columns built in notebook 01
for split in splits:
    assert (regroup_predictions(Y[split]["labels_28"], EKMAN_MAPPING) == Y[split]["labels_ekman"]).all()
    assert (regroup_predictions(Y[split]["labels_28"], SENTIMENT_MAPPING) == Y[split]["labels_sentiment"]).all()
print("2.1 OK - regrouping matches the prepared labels")

# 2.2 A perfect model scores 1 everywhere
m = compute_metrics(y_val, y_val)
assert all(abs(v - 1) < 1e-12 for v in m.values())
print("2.2 OK - perfect predictions:", m)

# 2.3 Our fast F1 (from TP / FP / FN counts, used by the bootstrap) equals scikit-learn's
rng = np.random.RandomState(0)
y_rand = (rng.rand(*y_val.shape) < 0.1).astype(int)
for avg in ["macro", "micro"]:
    tp = (y_val & y_rand).sum(axis = 0)
    fp = ((1 - y_val) & y_rand).sum(axis = 0)
    fn = (y_val & (1 - y_rand)).sum(axis = 0)
    ours = float(f1_from_counts(tp[None, :], fp[None, :], fn[None, :], avg)[0])
    sk = f1_score(y_val, y_rand, average = avg, zero_division = 0)
    assert abs(ours - sk) < 1e-12, (avg, ours, sk)
print("2.3 OK - bootstrap F1 matches scikit-learn")

# 2.4 The confidence interval contains the point estimate
v, lo, hi = bootstrap_ci(y_val, y_rand, "macro")
vm, lom, him = bootstrap_ci(y_val, y_rand, "micro")
assert lo <= v <= hi and lom <= vm <= him
print(f"2.4 OK - random predictions: macro-F1 {v:.3f} [{lo:.3f}, {hi:.3f}], micro-F1 {vm:.3f} [{lom:.3f}, {him:.3f}]")

# 2.5 apply_thresholds always predicts at least one label
proba_rand = rng.rand(*y_val.shape)
assert (apply_thresholds(proba_rand, np.full(28, 0.99)).sum(axis = 1) >= 1).all()
print("2.5 OK - at least one label per comment")

## 3. How to use the tools with a real model

The cell below uses a **fake** model (true labels + noise) only to show the workflow. Replace `proba_val` / `proba_test` by your model's `predict_proba` outputs (shape `(n_comments, 28)`, columns in `LABELS_28` order).

In [ ]:
# fake model: noisy version of the truth (for illustration only)
rng = np.random.RandomState(1)
def fake_proba(y, noise):
    return np.clip(0.6 * y + noise * rng.rand(*y.shape), 0, 1)

proba_val = fake_proba(Y["validation"]["labels_28"], 0.7)
y_true_val = {level: Y["validation"][level] for level in LEVELS}

# 1) tune one threshold per class on validation
thresholds = tune_thresholds(Y["validation"]["labels_28"], proba_val)

# 2) predict and score (here on validation; on test only for the final evaluation)
pred_val = apply_thresholds(proba_val, thresholds)
results = evaluate_all_levels("fake model", pred_val, y_true_val)
display(format_results(results)[["model", "level", "macro_f1", "micro_f1"]])

# 3) per-class view
report = per_class_report(Y["validation"]["labels_28"], pred_val, LABELS_28)
display(report.head())
plot_per_class_f1(report, "F1 per class (fake model, validation)");

### Comparing two models

Use the paired bootstrap rather than checking whether two CIs overlap: both models are scored on the same resamples, so the comparison is much more precise.

In [ ]:
pred_val_worse = apply_thresholds(fake_proba(Y["validation"]["labels_28"], 0.9), thresholds)
paired_bootstrap(Y["validation"]["labels_28"], pred_val_worse, pred_val)   # B = second model

### Calibration

In [ ]:
print(brier_per_label(Y["validation"]["labels_28"], proba_val, LABELS_28).head())
plot_reliability(Y["validation"]["labels_28"], proba_val, LABELS_28, ["neutral", "admiration", "gratitude", "grief"]);

## 4. Robustness suite

Hand-written messages grouped by difficulty (negation, sarcasm, very short texts, mixed emotions...). A message counts as a hit when **at least one** expected label is predicted. The expected labels are a first draft: **the team should review them before using the suite**.

In [ ]:
suite = pd.read_csv(os.path.join("..", "src", "robustness_suite.csv"))
all_expected = {lab for e in suite["expected"] for lab in e.split(";")}
assert all_expected <= set(LABELS_28), all_expected - set(LABELS_28)
print(len(suite), "messages")
print(suite["category"].value_counts())

# with a real model:
# proba_suite = model.predict_proba(vectorizer.transform(suite["text"]))
# pred_suite = apply_thresholds(proba_suite, thresholds)
# predicted = [[LABELS_28[j] for j in np.where(row)[0]] for row in pred_suite]
# detail, summary = evaluate_robustness(suite, predicted)

## 5. Summary

- All checks pass: regrouping matches the prepared labels, the bootstrap F1 equals scikit-learn's, a perfect model scores 1.
- Rules for every model: tune thresholds (and any calibration) on validation, compare models with `paired_bootstrap`, touch the test set once at the end.